In [10]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.news import (
    load_analyst_ratings_news,
    filter_news_to_universe,
    aggregate_daily_news,
    merge_news_with_price_panel,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
price_panel = pd.read_csv(PROJECT_ROOT / "data/processed/price_panel.csv")
price_panel["date"] = pd.to_datetime(price_panel["date"])

news_path = PROJECT_ROOT / "data/raw/analyst_ratings_processed.csv"

news = load_analyst_ratings_news(news_path)

news.head()

,date,ticker,text
0,2020-06-05,A,Stocks That Hit 52-Week Highs On Friday
1,2020-06-03,A,Stocks That Hit 52-Week Highs On Wednesday
2,2020-05-26,A,71 Biggest Movers From Friday
3,2020-05-22,A,46 Stocks Moving In Friday's Mid-Day Session
4,2020-05-22,A,B of A Securities Maintains Neutral on Agilent...


In [12]:
tickers = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META",
    "NVDA", "JPM", "BAC", "XOM", "CVX",
    "UNH", "JNJ", "PG", "KO", "PEP",
    "WMT", "HD", "COST", "DIS", "NFLX",
    "TSLA", "AMD", "INTC", "ORCL", "CRM",
    "GS", "MS", "V", "MA", "ADBE"
]

news_small = filter_news_to_universe(
    news=news,
    tickers=tickers,
    start_date="2020-01-01",
    end_date="2024-01-01"
)

news_small.shape

(5495, 3)

In [14]:
news_daily = aggregate_daily_news(
    news_small,
    max_items_per_day=10
)

news_daily.shape

(1633, 4)

In [15]:
news_panel = merge_news_with_price_panel(
    news_daily=news_daily,
    price_panel=price_panel
)

news_panel.shape

(1487, 10)

In [16]:
news_panel.to_csv(
    PROJECT_ROOT / "data/processed/news_price_panel.csv",
    index=False
)

In [17]:
news_panel[["date", "ticker", "text", "fwd_ret_1d", "fwd_ret_5d"]].head()

,date,ticker,text,fwd_ret_1d,fwd_ret_5d
0,2020-03-09,AAPL,Crude Awakening: Energy Sector Takes A 20% Spi...,0.072021,-0.090018
1,2020-03-10,AAPL,Peloton Shares Tick To Session Low As Hearing ...,-0.034731,-0.113829
2,2020-03-11,AAPL,Here's How Large Option Traders Are Playing Hi...,-0.098755,-0.104419
3,2020-03-12,AAPL,Selling Picks Up In Worst Day Since 1987 As Fe...,0.119808,-0.013898
4,2020-03-13,AAPL,Canopy Growth's Storz & Bickel Bypasses Apple'...,-0.128647,-0.175307


In [18]:
news_panel["ticker"].value_counts().head(10)

ticker
TSLA     111
GOOGL    110
NFLX     102
NVDA      96
XOM       91
BAC       89
AMD       88
JNJ       87
MS        80
MA        76
Name: count, dtype: int64

In [19]:
news_panel["date"].min(), news_panel["date"].max()

(Timestamp('2020-01-02 00:00:00'), Timestamp('2020-06-11 00:00:00'))